# 08 Evaluation Summaries and Report Tables

This notebook creates the final statistical summaries and report tables for the revised evaluation protocol.

- **Broaching:** primary fixed train/validation/test evaluation. Bootstrap 95% confidence intervals are computed on the independent test set.
- **Turning:** run-wise grouped five-fold cross-validation. Results are summarized as mean ± standard deviation across the five outer folds. Sample-level bootstrap intervals are intentionally not used because multiple windows originate from the same machining runs.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


In [11]:
REPO_ROOT = Path("..").resolve()
SCORE_DIR = REPO_ROOT / "reports" / "scores"
BASELINE_DIR = REPO_ROOT / "reports" / "baselines"
THRESHOLD_DIR = REPO_ROOT / "reports" / "thresholds"
TABLE_DIR = REPO_ROOT / "reports" / "tables"

TABLE_DIR.mkdir(parents=True, exist_ok=True)

N_BOOTSTRAP = 1000
RANDOM_SEED = 42

# Final Broaching artifacts
BROACH_AE_SCORES_PATH = SCORE_DIR / "ae_scores_broach_dataset.csv"
BROACH_AE_THRESHOLDS_PATH = THRESHOLD_DIR / "ae_thresholds_broach_dataset.json"
BROACH_BASELINE_SCORES_PATH = BASELINE_DIR / "baseline_scores_broaching_fixedsplit_rgb.csv"
BROACH_BASELINE_THRESHOLDS_PATH = THRESHOLD_DIR / "baseline_thresholds_broaching_fixedsplit_rgb.json"

# Final Turning grouped-CV artifacts
TURNING_AE_METRICS_PATH = TABLE_DIR / "metrics_turning_ae_grouped_cv.csv"
TURNING_BASELINE_METRICS_PATH = TABLE_DIR / "metrics_turning_grouped_cv_rgb.csv"

BROACH_AE_SCORE_COLUMNS = ["global_mse", "global_mae", "sve_5", "sve_7"]


## Shared metric helpers

Bootstrap intervals are used only for the independent Broaching test set. Turning uncertainty is represented by fold-to-fold standard deviation from the grouped five-fold evaluation.


In [3]:
def evaluate_at_threshold(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "pr_auc": float(average_precision_score(y_true, scores)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def bootstrap_intervals(
    y_true: np.ndarray,
    scores: np.ndarray,
    threshold: float,
    n_bootstrap: int = N_BOOTSTRAP,
    seed: int = RANDOM_SEED,
) -> dict:
    rng = np.random.default_rng(seed)
    n = len(y_true)
    rows = []

    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, size=n)
        sampled_y = y_true[idx]
        sampled_scores = scores[idx]

        if len(np.unique(sampled_y)) < 2:
            continue

        rows.append(
            evaluate_at_threshold(sampled_y, sampled_scores, threshold)
        )

    boot = pd.DataFrame(rows)
    if boot.empty:
        raise ValueError("No valid bootstrap samples were generated.")

    intervals = {}
    for metric in ["pr_auc", "f1", "precision", "recall"]:
        intervals[f"{metric}_ci_low"] = float(boot[metric].quantile(0.025))
        intervals[f"{metric}_ci_high"] = float(boot[metric].quantile(0.975))

    intervals["bootstrap_n_effective"] = int(len(boot))
    return intervals


def format_ci(value, low, high):
    return f"{value:.3f} [{low:.3f}, {high:.3f}]"


def format_mean_sd(mean, std):
    return f"{mean:.3f} ± {std:.3f}"


## Broaching: CNN-AE fixed-split evaluation

The CNN-AE threshold is taken from the validation set and applied unchanged to the independent test set. Bootstrap confidence intervals are calculated from the held-out test samples.


In [4]:
if not BROACH_AE_SCORES_PATH.exists():
    raise FileNotFoundError(BROACH_AE_SCORES_PATH)
if not BROACH_AE_THRESHOLDS_PATH.exists():
    raise FileNotFoundError(BROACH_AE_THRESHOLDS_PATH)

broach_ae_scores = pd.read_csv(BROACH_AE_SCORES_PATH)
with BROACH_AE_THRESHOLDS_PATH.open() as f:
    broach_ae_thresholds = json.load(f)

broach_ae_test = broach_ae_scores[broach_ae_scores["split"] == "test"].copy()

# Broaching manifests use binary labels. Prefer target if it is already available.
if "target" in broach_ae_test.columns:
    y_true = broach_ae_test["target"].astype(int).to_numpy()
else:
    y_true = broach_ae_test["label"].astype(int).to_numpy()

rows = []
for score_name in BROACH_AE_SCORE_COLUMNS:
    if score_name not in broach_ae_thresholds or score_name not in broach_ae_test.columns:
        continue

    threshold = float(broach_ae_thresholds[score_name]["threshold"])
    score_values = broach_ae_test[score_name].to_numpy()

    metrics = evaluate_at_threshold(y_true, score_values, threshold)
    intervals = bootstrap_intervals(y_true, score_values, threshold)

    rows.append({
        "dataset": "broach_dataset",
        "method": "cnn_ae",
        "score": score_name,
        "threshold": threshold,
        **metrics,
        **intervals,
    })

broach_ae_ci = pd.DataFrame(rows)
BROACH_AE_CI_PATH = TABLE_DIR / "metrics_broaching_ae_fixedsplit_with_ci.csv"
broach_ae_ci.to_csv(BROACH_AE_CI_PATH, index=False)
print(f"Wrote {BROACH_AE_CI_PATH}")
display(broach_ae_ci)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broaching_ae_fixedsplit_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,broach_dataset,cnn_ae,global_mse,0.001131,0.592931,0.666667,1.000000,0.500000,2478,0,...,11,0.394239,0.771255,0.444444,0.823529,1.000000,1.0,0.285714,0.700000,1000
1,broach_dataset,cnn_ae,global_mae,0.013949,0.465159,0.516129,0.888889,0.363636,2477,1,...,8,0.273575,0.650260,0.272727,0.705977,0.636364,1.0,0.166667,0.571429,1000
2,broach_dataset,cnn_ae,sve_5,0.001871,0.945071,0.909091,0.909091,0.909091,2476,2,...,20,0.846621,1.000000,0.800000,0.980392,0.764706,1.0,0.761905,1.000000,1000
3,broach_dataset,cnn_ae,sve_7,0.003109,0.979270,0.850000,0.944444,0.772727,2477,1,...,17,0.934970,1.000000,0.717857,0.945455,0.823396,1.0,0.590909,0.928571,1000


## Broaching: RGB baselines on the same fixed split

PCA, one-class SVM and isolation forest use the revised three-channel RGB representation. Their thresholds were calibrated on the same validation split and applied unchanged to the same independent test split.


In [5]:
if not BROACH_BASELINE_SCORES_PATH.exists():
    raise FileNotFoundError(BROACH_BASELINE_SCORES_PATH)
if not BROACH_BASELINE_THRESHOLDS_PATH.exists():
    raise FileNotFoundError(BROACH_BASELINE_THRESHOLDS_PATH)

broach_baseline_scores = pd.read_csv(BROACH_BASELINE_SCORES_PATH)
with BROACH_BASELINE_THRESHOLDS_PATH.open() as f:
    broach_baseline_thresholds = json.load(f)

rows = []
for method, group in broach_baseline_scores.groupby("method"):
    # 06c exports the independent test-set scores only. The filter is kept if
    # a split column is present for compatibility with existing reports.
    if "split" in group.columns:
        group = group[group["split"] == "test"].copy()

    threshold_entry = broach_baseline_thresholds[method]
    threshold = float(
        threshold_entry["threshold"]
        if isinstance(threshold_entry, dict)
        else threshold_entry
    )

    y_true = group["target"].astype(int).to_numpy()
    score_values = group["score_value"].to_numpy()

    metrics = evaluate_at_threshold(y_true, score_values, threshold)
    intervals = bootstrap_intervals(y_true, score_values, threshold)

    rows.append({
        "dataset": "broach_dataset",
        "method": method,
        "score": "reconstruction_mse" if method == "pca_rgb_reconstruction" else "anomaly_score",
        "threshold": threshold,
        **metrics,
        **intervals,
    })

broach_baseline_ci = pd.DataFrame(rows)
BROACH_BASELINE_CI_PATH = TABLE_DIR / "metrics_broaching_baselines_fixedsplit_rgb_with_ci.csv"
broach_baseline_ci.to_csv(BROACH_BASELINE_CI_PATH, index=False)
print(f"Wrote {BROACH_BASELINE_CI_PATH}")
display(broach_baseline_ci)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broaching_baselines_fixedsplit_rgb_with_ci.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,...,tp,pr_auc_ci_low,pr_auc_ci_high,f1_ci_low,f1_ci_high,precision_ci_low,precision_ci_high,recall_ci_low,recall_ci_high,bootstrap_n_effective
0,broach_dataset,isolation_forest_rgb_features,anomaly_score,0.152294,0.640653,0.530612,0.481481,0.590909,2464,14,...,13,0.441619,0.802482,0.355556,0.681818,0.304315,0.677484,0.400000,0.789474,1000
1,broach_dataset,one_class_svm_rgb_features,anomaly_score,14.167647,0.838384,0.700000,0.777778,0.636364,2474,4,...,14,0.698159,0.944566,0.514286,0.850027,0.562500,0.952435,0.420943,0.833468,1000
2,broach_dataset,pca_rgb_reconstruction,reconstruction_mse,0.000668,0.982175,0.842105,1.000000,0.727273,2478,0,...,16,0.944423,1.000000,0.689655,0.950000,1.000000,1.000000,0.526316,0.904762,1000


## Broaching report tables

Separate compact tables are exported for the CNN-AE scores and the classical RGB baselines. Both are based on the same independent fixed test split.


In [6]:
broach_ae_labels = {
    "global_mse": "Global MSE",
    "global_mae": "Global MAE",
    "sve_5": "SVE 5 Segments",
    "sve_7": "SVE 7 Segments",
}

broach_ae_report = broach_ae_ci.copy()
broach_ae_report["Method"] = broach_ae_report["score"].map(broach_ae_labels)

for metric, title in [
    ("pr_auc", "PR-AUC [95% CI]"),
    ("f1", "F1-score [95% CI]"),
    ("precision", "Precision [95% CI]"),
    ("recall", "Recall [95% CI]"),
]:
    broach_ae_report[title] = broach_ae_report.apply(
        lambda r, m=metric: format_ci(
            r[m], r[f"{m}_ci_low"], r[f"{m}_ci_high"]
        ),
        axis=1,
    )

broach_ae_report = broach_ae_report[[
    "Method",
    "PR-AUC [95% CI]",
    "F1-score [95% CI]",
    "Precision [95% CI]",
    "Recall [95% CI]",
]]

BROACH_AE_REPORT_PATH = TABLE_DIR / "report_broaching_ae_fixedsplit.csv"
broach_ae_report.to_csv(BROACH_AE_REPORT_PATH, index=False)
print(f"Wrote {BROACH_AE_REPORT_PATH}")
display(broach_ae_report)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\report_broaching_ae_fixedsplit.csv


,Method,PR-AUC [95% CI],F1-score [95% CI],Precision [95% CI],Recall [95% CI]
0,Global MSE,"0.593 [0.394, 0.771]","0.667 [0.444, 0.824]","1.000 [1.000, 1.000]","0.500 [0.286, 0.700]"
1,Global MAE,"0.465 [0.274, 0.650]","0.516 [0.273, 0.706]","0.889 [0.636, 1.000]","0.364 [0.167, 0.571]"
2,SVE 5 Segments,"0.945 [0.847, 1.000]","0.909 [0.800, 0.980]","0.909 [0.765, 1.000]","0.909 [0.762, 1.000]"
3,SVE 7 Segments,"0.979 [0.935, 1.000]","0.850 [0.718, 0.945]","0.944 [0.823, 1.000]","0.773 [0.591, 0.929]"


In [7]:
baseline_labels = {
    "one_class_svm_rgb_features": "One-Class SVM",
    "isolation_forest_rgb_features": "Isolation Forest",
    "pca_rgb_reconstruction": "PCA Reconstruction",
}

broach_baseline_report = broach_baseline_ci.copy()
broach_baseline_report["Method"] = broach_baseline_report["method"].map(baseline_labels)

for metric, title in [
    ("pr_auc", "PR-AUC [95% CI]"),
    ("f1", "F1-score [95% CI]"),
    ("precision", "Precision [95% CI]"),
    ("recall", "Recall [95% CI]"),
]:
    broach_baseline_report[title] = broach_baseline_report.apply(
        lambda r, m=metric: format_ci(
            r[m], r[f"{m}_ci_low"], r[f"{m}_ci_high"]
        ),
        axis=1,
    )

broach_baseline_report = broach_baseline_report[[
    "Method",
    "PR-AUC [95% CI]",
    "F1-score [95% CI]",
    "Precision [95% CI]",
    "Recall [95% CI]",
]]

BROACH_BASELINE_REPORT_PATH = TABLE_DIR / "report_broaching_baselines_fixedsplit_rgb.csv"
broach_baseline_report.to_csv(BROACH_BASELINE_REPORT_PATH, index=False)
print(f"Wrote {BROACH_BASELINE_REPORT_PATH}")
display(broach_baseline_report)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\report_broaching_baselines_fixedsplit_rgb.csv


,Method,PR-AUC [95% CI],F1-score [95% CI],Precision [95% CI],Recall [95% CI]
0,Isolation Forest,"0.641 [0.442, 0.802]","0.531 [0.356, 0.682]","0.481 [0.304, 0.677]","0.591 [0.400, 0.789]"
1,One-Class SVM,"0.838 [0.698, 0.945]","0.700 [0.514, 0.850]","0.778 [0.562, 0.952]","0.636 [0.421, 0.833]"
2,PCA Reconstruction,"0.982 [0.944, 1.000]","0.842 [0.690, 0.950]","1.000 [1.000, 1.000]","0.727 [0.526, 0.905]"


## Turning: CNN-AE grouped five-fold cross-validation

Turning results are summarized over the five run-wise grouped outer folds. Mean ± standard deviation is reported instead of sample-level bootstrap confidence intervals.


In [9]:
if not TURNING_AE_METRICS_PATH.exists():
    raise FileNotFoundError(TURNING_AE_METRICS_PATH)

turning_ae_metrics = pd.read_csv(TURNING_AE_METRICS_PATH)

assert turning_ae_metrics["outer_fold"].nunique() == 5

turning_ae_summary = (
    turning_ae_metrics
    .groupby("score")
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
    )
    .reset_index()
)

TURNING_AE_SUMMARY_PATH = TABLE_DIR / "metrics_turning_ae_grouped_cv_summary.csv"
turning_ae_summary.to_csv(TURNING_AE_SUMMARY_PATH, index=False)
print(f"Wrote {TURNING_AE_SUMMARY_PATH}")
display(turning_ae_summary)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_ae_grouped_cv_summary.csv


,score,pr_auc_mean,pr_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,global_mae,0.81922,0.229639,0.63354,0.266961,0.69888,0.139791,0.65740,0.387110
1,global_mse,0.78790,0.221130,0.60276,0.268585,0.63306,0.202092,0.65740,0.387110
2,sve_max,0.77474,0.215317,0.63282,0.224405,0.70540,0.222059,0.73064,0.353686
3,sve_topk,0.78756,0.219035,0.58118,0.263113,0.59192,0.222836,0.65740,0.387110


## Turning: RGB baselines using the same grouped folds

The classical baselines use the same run-wise outer-fold assignments and the same group-wise calibration protocol as the CNN-AE.


In [12]:
if not TURNING_BASELINE_METRICS_PATH.exists():
    raise FileNotFoundError(TURNING_BASELINE_METRICS_PATH)

turning_baseline_metrics = pd.read_csv(TURNING_BASELINE_METRICS_PATH)

assert turning_baseline_metrics["outer_fold"].nunique() == 5

turning_baseline_summary = (
    turning_baseline_metrics
    .groupby("method")
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
    )
    .reset_index()
)

TURNING_BASELINE_SUMMARY_PATH = TABLE_DIR / "metrics_turning_baselines_grouped_cv_rgb_summary.csv"
turning_baseline_summary.to_csv(TURNING_BASELINE_SUMMARY_PATH, index=False)
print(f"Wrote {TURNING_BASELINE_SUMMARY_PATH}")
display(turning_baseline_summary)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_baselines_grouped_cv_rgb_summary.csv


,method,pr_auc_mean,pr_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,isolation_forest_rgb_features,0.881970,0.137520,0.746813,0.192833,0.878914,0.132052,0.716450,0.279398
1,one_class_svm_rgb_features,0.753695,0.172332,0.646937,0.169438,0.701966,0.230812,0.742165,0.315429
2,pca_rgb_reconstruction,0.723586,0.192068,0.613999,0.165428,0.661354,0.233866,0.708831,0.307694


## Turning report table

The table combines CNN-AE reconstruction scores and RGB classical baselines under the same run-wise grouped five-fold evaluation. Values are reported as mean ± standard deviation over the five outer folds.


In [13]:
ae_labels = {
    "global_mse": "CNN-AE Global MSE",
    "global_mae": "CNN-AE Global MAE",
    "sve_max": "CNN-AE SVE Max",
    "sve_topk": "CNN-AE SVE Top-k",
}

turning_ae_report = turning_ae_summary.copy()
turning_ae_report["Method"] = turning_ae_report["score"].map(ae_labels)

turning_baseline_report = turning_baseline_summary.copy()
turning_baseline_report["Method"] = turning_baseline_report["method"].map(baseline_labels)

summary_columns = [
    "pr_auc_mean", "pr_auc_std",
    "f1_mean", "f1_std",
    "precision_mean", "precision_std",
    "recall_mean", "recall_std",
]

turning_report = pd.concat([
    turning_ae_report[["Method", *summary_columns]],
    turning_baseline_report[["Method", *summary_columns]],
], ignore_index=True)

turning_report["PR-AUC (mean ± SD)"] = turning_report.apply(
    lambda r: format_mean_sd(r["pr_auc_mean"], r["pr_auc_std"]), axis=1
)
turning_report["F1-score (mean ± SD)"] = turning_report.apply(
    lambda r: format_mean_sd(r["f1_mean"], r["f1_std"]), axis=1
)
turning_report["Precision (mean ± SD)"] = turning_report.apply(
    lambda r: format_mean_sd(r["precision_mean"], r["precision_std"]), axis=1
)
turning_report["Recall (mean ± SD)"] = turning_report.apply(
    lambda r: format_mean_sd(r["recall_mean"], r["recall_std"]), axis=1
)

turning_report = turning_report[[
    "Method",
    "PR-AUC (mean ± SD)",
    "F1-score (mean ± SD)",
    "Precision (mean ± SD)",
    "Recall (mean ± SD)",
]]

TURNING_REPORT_PATH = TABLE_DIR / "report_turning_grouped_cv.csv"
turning_report.to_csv(TURNING_REPORT_PATH, index=False)
print(f"Wrote {TURNING_REPORT_PATH}")
display(turning_report)


Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\report_turning_grouped_cv.csv


,Method,PR-AUC (mean ± SD),F1-score (mean ± SD),Precision (mean ± SD),Recall (mean ± SD)
0,CNN-AE Global MAE,0.819 ± 0.230,0.634 ± 0.267,0.699 ± 0.140,0.657 ± 0.387
1,CNN-AE Global MSE,0.788 ± 0.221,0.603 ± 0.269,0.633 ± 0.202,0.657 ± 0.387
2,CNN-AE SVE Max,0.775 ± 0.215,0.633 ± 0.224,0.705 ± 0.222,0.731 ± 0.354
3,CNN-AE SVE Top-k,0.788 ± 0.219,0.581 ± 0.263,0.592 ± 0.223,0.657 ± 0.387
4,Isolation Forest,0.882 ± 0.138,0.747 ± 0.193,0.879 ± 0.132,0.716 ± 0.279
5,One-Class SVM,0.754 ± 0.172,0.647 ± 0.169,0.702 ± 0.231,0.742 ± 0.315
6,PCA Reconstruction,0.724 ± 0.192,0.614 ± 0.165,0.661 ± 0.234,0.709 ± 0.308


## Final artifact checks

The final report structure intentionally reflects the two different evaluation designs:

- Broaching: fixed split + bootstrap 95% confidence intervals on the independent test set.
- Turning: run-wise grouped five-fold CV + mean ± SD across outer folds.


In [14]:
expected_outputs = [
    BROACH_AE_CI_PATH,
    BROACH_BASELINE_CI_PATH,
    BROACH_AE_REPORT_PATH,
    BROACH_BASELINE_REPORT_PATH,
    TURNING_AE_SUMMARY_PATH,
    TURNING_BASELINE_SUMMARY_PATH,
    TURNING_REPORT_PATH,
]

for path in expected_outputs:
    assert path.exists(), f"Missing output: {path}"

print("All evaluation summary artifacts were written successfully.")
for path in expected_outputs:
    print(f"  {path}")


All evaluation summary artifacts were written successfully.
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broaching_ae_fixedsplit_with_ci.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broaching_baselines_fixedsplit_rgb_with_ci.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\report_broaching_ae_fixedsplit.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\report_broaching_baselines_fixedsplit_rgb.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_ae_grouped_cv_summary.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_turning_baselines_grouped_cv_rgb_summary.csv
  C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\report_turning_grouped_cv.csv
